<a href="https://colab.research.google.com/github/miaflynn/CYPLAN255-Final-Project/blob/main/03d_visualizations_survival_rate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Analysis still in progress

##Importing

In [42]:
import importlib
importlib.reload(functions)
import pandas as pd
import geopandas as gpd
import numpy as np
import os
%matplotlib inline
import matplotlib.pyplot as plt
from shapely.geometry import LineString
import plotly.express as px

import sys
sys.path.append('/content')

import functions

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:

!ls /content/drive
!ls /content/drive/Shareddrives

MyDrive  Shareddrives


##Reading in file

In [19]:
gdf = gpd.read_parquet("/content/drive/MyDrive/C255_final_project/cleaned/rbl_GEOID_all_dates.parquet")

##Reading in Census Tracts

In [20]:
sf_tracts = gpd.read_parquet("/content/drive/MyDrive/C255_final_project/cleaned/sf_tracts.parquet")

In [50]:
gdf['location_end_date'] = pd.to_datetime(gdf['location_end_date'])
gdf['location_start_date'] = pd.to_datetime(gdf['location_start_date'])
gdf['open_month_year'] = gdf['location_start_date'].dt.strftime('%B %Y')
gdf['close_month_year'] = gdf['location_end_date'].dt.strftime('%B %Y')

In [58]:

pre_covid_mask = gdf['location_start_date'] < '2020-03-01'
pre_covid_tracts = gdf[pre_covid_mask]

survival_mask = (gdf['location_start_date'].dt.year < 2020) & gdf['location_end_date'].isna()
survival_tracts = gdf[survival_mask].sort_values('location_start_date')

In [62]:
pre_covid_tracts_grouped = functions.group_points_by_tract(
    points=pre_covid_tracts,
    tracts=sf_tracts
)

survival_tracts_grouped = functions.group_points_by_tract(
    points=survival_tracts,
    tracts=sf_tracts
)

# adding closed businesses and opened businesses that have not closed
# (because closed businesses also are listed in opened, just in a diff year)
survival_tracts_grouped['total_businesses'] = pre_covid_tracts_grouped['closed'] + survival_tracts_grouped['opened']

In [70]:
import plotly.graph_objects as go

fig = px.choropleth_mapbox(
    survival_tracts_grouped,
    geojson=survival_tracts_grouped.set_index("GEOID").__geo_interface__,
    locations="GEOID",
    color="survival_rate",
    hover_name="GEOID",
    center={"lat": 37.7749, "lon": -122.4194},
    zoom=10,
    mapbox_style="carto-positron",
    color_continuous_scale="Reds",
    height=500,
    width=700

)


fig.show()